In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### **Importing Prerequisites**

In [0]:
%run "/Workspace/Users/0bcr9999@gmail.com/FMGC-DATAENGINEERING-PROJECT/notebooks/setup/3. utilities"

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

### **Reading Raw data from Source**

In [0]:
orders_base_path = f"/Volumes/fmcg/default/child_company/fullload/full_load/orders"
orders_landing_path = f"/Volumes/fmcg/default/child_company/fullload/full_load/orders/landing/"
orders_processed_path = f"/Volumes/fmcg/default/child_company/fullload/full_load/orders/processed/"
dbutils.fs.ls(orders_base_path)

In [0]:
df_bronze = spark.read.format("csv") \
    .option("header", True) \
        .option("inferSchema", True) \
            .load(f"{orders_landing_path}*.csv")

In [0]:
display(df_bronze.limt(10))

### **Addition of metadata info**

In [0]:
df_bronze = df_bronze.withColumn("dataload_ts", F.current_timestamp()) \
  .select("*", "_metadata.file_name", "_metadata.file_size")

In [0]:
display(df_bronze.limit(10))

In [0]:
df_bronze.printSchema()

In [0]:
df_bronze.write \
    .format("delta") \
        .option("enable.changeChangeDataFeed", True) \
            .mode("append") \
                .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
file = dbutils.fs.ls(orders_landing_path)

for f in file:
    dbutils.fs.cp(f.path, f"{orders_processed_path}/{f.name}", True)

In [0]:
print(len(dbutils.fs.ls(orders_landing_path)),
len(dbutils.fs.ls(orders_processed_path)))